# Design Patterns in Python

**OOP-in-Python — Advanced**  
Singleton, Factory, Observer, Strategy, Decorator, and Builder patterns with Pythonic implementations.

## 1. Singleton Pattern

Ensures a class has **only one instance** throughout the application.

Python approaches:
1. **Module-level** — simplest (modules are singletons by nature)
2. **`__new__` override** — classic OOP approach
3. **Metaclass** — most reusable

In [ ]:
# Approach 1: Module-level singleton (simplest)
# In a real project, put this in a module and import it.
class _Config:
    def __init__(self):
        self.settings = {"debug": False, "db_url": "sqlite:///app.db"}

config = _Config()  # Single instance at module level

# Approach 2: __new__ based Singleton
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        pass  # __init__ runs every time, so be careful

s1 = Singleton()
s2 = Singleton()
print(s1 is s2)  # True

In [ ]:
# Approach 3: Metaclass-based Singleton (most reusable)
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Logger(metaclass=SingletonMeta):
    def __init__(self):
        self.logs = []

    def log(self, msg: str):
        self.logs.append(msg)
        print(f"[LOG] {msg}")

l1 = Logger()
l2 = Logger()
print(l1 is l2)  # True
l1.log("First message")
print(len(l2.logs))  # 1 (same instance)

## 2. Factory Pattern

Creates objects **without specifying the exact class** — useful when object creation is complex or conditional.

In [ ]:
from abc import ABC, abstractmethod

class Animal(ABC):
    @abstractmethod
    def speak(self) -> str: ...

class Dog(Animal):
    def speak(self) -> str:
        return "Woof"

class Cat(Animal):
    def speak(self) -> str:
        return "Meow"

class Bird(Animal):
    def speak(self) -> str:
        return "Chirp"

class AnimalFactory:
    @staticmethod
    def create(animal_type: str) -> Animal:
        factories = {
            "dog": Dog,
            "cat": Cat,
            "bird": Bird,
        }
        cls = factories.get(animal_type.lower())
        if cls is None:
            raise ValueError(f"Unknown animal: {animal_type}")
        return cls()

# Usage — client doesn't care about concrete class
animals = [AnimalFactory.create("dog"), AnimalFactory.create("cat")]
for a in animals:
    print(a.speak())

## 3. Observer Pattern

Defines a **one-to-many** dependency: when one object changes state, all its dependents are notified automatically.

In [ ]:
from abc import ABC, abstractmethod

class Observer(ABC):
    @abstractmethod
    def update(self, event_type: str, data): ...

class Subject:
    def __init__(self):
        self._observers = []

    def attach(self, observer: Observer):
        self._observers.append(observer)

    def detach(self, observer: Observer):
        self._observers.remove(observer)

    def notify(self, event_type: str, data=None):
        for obs in self._observers:
            obs.update(event_type, data)

class EmailNotifier(Observer):
    def update(self, event_type: str, data):
        print(f"[EMAIL] Event '{event_type}': {data} — Sending email...")

class SMSNotifier(Observer):
    def update(self, event_type: str, data):
        print(f"[SMS] Event '{event_type}': {data} — Sending SMS...")

class LogNotifier(Observer):
    def update(self, event_type: str, data):
        print(f"[LOG] Event '{event_type}': {data}")

# Usage
subject = Subject()
subject.attach(EmailNotifier())
subject.attach(SMSNotifier())
subject.attach(LogNotifier())

subject.notify("user_registered", {"email": "alice@example.com"})
print("---")
subject.notify("order_shipped", {"order_id": 12345})

## 4. Strategy Pattern

Defines a family of algorithms, encapsulates each, and makes them interchangeable.

In [ ]:
from abc import ABC, abstractmethod
from typing import List

class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data: List[int]) -> List[int]: ...

class BubbleSort(SortStrategy):
    def sort(self, data: List[int]) -> List[int]:
        arr = data.copy()
        n = len(arr)
        for i in range(n):
            for j in range(0, n - i - 1):
                if arr[j] > arr[j + 1]:
                    arr[j], arr[j + 1] = arr[j + 1], arr[j]
        return arr

class QuickSort(SortStrategy):
    def sort(self, data: List[int]) -> List[int]:
        if len(data) <= 1:
            return data
        pivot = data[0]
        left = [x for x in data[1:] if x <= pivot]
        right = [x for x in data[1:] if x > pivot]
        return self.sort(left) + [pivot] + self.sort(right)

class Sorter:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy

    def set_strategy(self, strategy: SortStrategy):
        self._strategy = strategy

    def sort(self, data: List[int]) -> List[int]:
        return self._strategy.sort(data)

data = [64, 34, 25, 12, 22, 11, 90]
sorter = Sorter(BubbleSort())
print("Bubble:", sorter.sort(data))
sorter.set_strategy(QuickSort())
print("Quick:", sorter.sort(data))

## 5. Decorator Pattern (GoF)

Attaches **additional responsibilities** to an object dynamically. (Not to be confused with Python's `@decorator` syntax — though the concept is similar.)

In [ ]:
from abc import ABC, abstractmethod

class Coffee(ABC):
    @abstractmethod
    def cost(self) -> float: ...

    @abstractmethod
    def description(self) -> str: ...

class BasicCoffee(Coffee):
    def cost(self) -> float:
        return 2.0

    def description(self) -> str:
        return "Basic coffee"

class CoffeeDecorator(Coffee, ABC):
    def __init__(self, coffee: Coffee):
        self._coffee = coffee

class MilkDecorator(CoffeeDecorator):
    def cost(self) -> float:
        return self._coffee.cost() + 0.5

    def description(self) -> str:
        return self._coffee.description() + " + milk"

class SugarDecorator(CoffeeDecorator):
    def cost(self) -> float:
        return self._coffee.cost() + 0.25

    def description(self) -> str:
        return self._coffee.description() + " + sugar"

class WhippedCreamDecorator(CoffeeDecorator):
    def cost(self) -> float:
        return self._coffee.cost() + 0.75

    def description(self) -> str:
        return self._coffee.description() + " + whipped cream"

# Build a custom coffee
coffee = BasicCoffee()
coffee = MilkDecorator(coffee)
coffee = SugarDecorator(coffee)
coffee = WhippedCreamDecorator(coffee)
print(f"{coffee.description()} — ${coffee.cost():.2f}")
# Basic coffee + milk + sugar + whipped cream — $3.50

## 6. Builder Pattern

Separates the construction of a complex object from its representation.

In [ ]:
class Pizza:
    def __init__(self):
        self.size = None
        self.crust = None
        self.toppings = []
        self.cheese = None
        self.sauce = None

    def __str__(self) -> str:
        return (f"{self.size} pizza ({self.crust} crust)"
                f" with {', '.join(self.toppings)}, {self.cheese} cheese"
                f" and {self.sauce} sauce")

class PizzaBuilder:
    def __init__(self):
        self.pizza = Pizza()

    def set_size(self, size: str) -> 'PizzaBuilder':
        self.pizza.size = size
        return self

    def set_crust(self, crust: str) -> 'PizzaBuilder':
        self.pizza.crust = crust
        return self

    def add_topping(self, topping: str) -> 'PizzaBuilder':
        self.pizza.toppings.append(topping)
        return self

    def set_cheese(self, cheese: str) -> 'PizzaBuilder':
        self.pizza.cheese = cheese
        return self

    def set_sauce(self, sauce: str) -> 'PizzaBuilder':
        self.pizza.sauce = sauce
        return self

    def build(self) -> Pizza:
        # Validate before returning
        assert self.pizza.size is not None, "Size required"
        assert self.pizza.crust is not None, "Crust required"
        return self.pizza

# Fluent interface — method chaining
pizza = (PizzaBuilder()
         .set_size("Large")
         .set_crust("Thin")
         .add_topping("Pepperoni")
         .add_topping("Mushrooms")
         .add_topping("Olives")
         .set_cheese("Mozzarella")
         .set_sauce("Tomato")
         .build())
print(pizza)

---
## Problem 1: Singleton Logger

**Difficulty: Easy**

Implement a thread-safe Singleton Logger using metaclass.

In [ ]:
import threading

class ThreadSafeSingletonMeta(type):
    _instances = {}
    _lock = threading.Lock()

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            with cls._lock:
                # Double-checked locking
                if cls not in cls._instances:
                    cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class AppLogger(metaclass=ThreadSafeSingletonMeta):
    def __init__(self):
        self.logs = []

    def info(self, msg: str):
        entry = f"[INFO] {msg}"
        self.logs.append(entry)
        print(entry)

    def error(self, msg: str):
        entry = f"[ERROR] {msg}"
        self.logs.append(entry)
        print(entry)

    def get_logs(self) -> list:
        return list(self.logs)

# Test in multiple threads
def worker(n: int):
    logger = AppLogger()
    logger.info(f"Worker {n} started")

threads = [threading.Thread(target=worker, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

logger = AppLogger()
print(f"Total log entries: {len(logger.get_logs())}")  # All 5 from same instance
print(f"All same instance? {logger is AppLogger()}")  # True

## Problem 2: Observer Pattern for Event System

**Difficulty: Medium**

Build a flexible event system where observers subscribe to specific event types.

In [ ]:
from collections import defaultdict

class EventBus:
    def __init__(self):
        self._listeners = defaultdict(list)  # event_type -> [callbacks]

    def subscribe(self, event_type: str, callback):
        self._listeners[event_type].append(callback)

    def unsubscribe(self, event_type: str, callback):
        self._listeners[event_type].remove(callback)

    def emit(self, event_type: str, **data):
        for callback in self._listeners.get(event_type, []):
            callback(event_type, data)

# Concrete observers
class AnalyticsTracker:
    def __init__(self):
        self.events = defaultdict(int)

    def track(self, event_type: str, data: dict):
        self.events[event_type] += 1
        print(f"[Analytics] Tracked '{event_type}' (total: {self.events[event_type]})")

class EmailService:
    def on_user_registered(self, event_type: str, data: dict):
        print(f"[Email] Sending welcome to {data.get('email', 'unknown')}")

    def on_password_reset(self, event_type: str, data: dict):
        print(f"[Email] Sending password reset to {data.get('email', 'unknown')}")

class AuditLogger:
    def log(self, event_type: str, data: dict):
        print(f"[Audit] {event_type}: {data}")

# Wire it up
bus = EventBus()
analytics = AnalyticsTracker()
email = EmailService()
audit = AuditLogger()

bus.subscribe("user.registered", analytics.track)
bus.subscribe("user.registered", email.on_user_registered)
bus.subscribe("user.registered", audit.log)
bus.subscribe("password.reset", email.on_password_reset)
bus.subscribe("password.reset", audit.log)

# Fire events
bus.emit("user.registered", email="alice@example.com", name="Alice")
print("---")
bus.emit("password.reset", email="bob@example.com")

## Key Takeaways

- **Singleton**: one instance — use metaclass or module-level for thread safety.
- **Factory**: centralize object creation — great for conditional/complex instantiation.
- **Observer**: one-to-many notification — decouples event sources from consumers.
- **Strategy**: interchangeable algorithms — composition over inheritance in action.
- **Decorator**: dynamically add behavior — wraps objects recursively.
- **Builder**: fluent step-by-step construction — excellent for complex objects.
- Design patterns are **guidelines**, not rules. Adapt them to Python's idioms.